# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [35]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

fatal: destination path 'SQuAD-explorer' already exists and is not an empty directory.


In [36]:
import json

In [37]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [38]:
type(train_data)

dict

In [39]:
list(train_data.keys())

['data', 'version']

In [40]:
type(train_data["data"])

list

In [41]:
len(train_data["data"])

442

In [42]:
type(train_data["data"][0])

dict

In [43]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [44]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [45]:
len(train_data["data"][0]["paragraphs"])

55

In [46]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [47]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [48]:
import pandas as pd

rows = []
for doc in train_data["data"]:
    title = doc["title"]
    for i, para in enumerate(doc["paragraphs"]):
        rows.append({
            "document_title": title,
            "paragraph_index": i,
            "paragraph_context": para["context"]
        })

parsed_df = pd.DataFrame(rows)
parsed_df.to_csv("parsed.tsv", sep="\t", index=False)
print(f"Saved parsed.tsv with {len(parsed_df)} rows")
parsed_df.head()

Saved parsed.tsv with 18896 rows


,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import numpy as np

# Build TF-IDF matrix from paragraph contexts
vectorizer = TfidfVectorizer(max_features=50000)
tfidf_matrix = vectorizer.fit_transform(parsed_df["paragraph_context"])

# Reduce to 1024 dimensions using LSA (TruncatedSVD)
svd = TruncatedSVD(n_components=1024, random_state=42)
paragraph_vectors = svd.fit_transform(tfidf_matrix)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Paragraph vectors shape: {paragraph_vectors.shape}")

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [ ]:
para_vec_df = parsed_df[["document_title", "paragraph_index"]].copy()
para_vec_df["paragraph_vector_json"] = [json.dumps([round(x, 4) for x in v.tolist()]) for v in paragraph_vectors]
para_vec_df.to_csv("paragraph-vectors.tsv.gz", sep="\t", index=False)
print(f"Saved paragraph-vectors.tsv.gz with {len(para_vec_df)} rows")

Saved paragraph-vectors.tsv.gz with 18896 rows


Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [ ]:
questions_df = pd.read_csv("questions.tsv", sep="\t")

# Encode questions using the same TF-IDF vectorizer and SVD from the paragraphs
question_tfidf = vectorizer.transform(questions_df["question"])
question_vectors = svd.transform(question_tfidf)

print(f"Questions shape: {questions_df.shape}")
print(f"Question vectors shape: {question_vectors.shape}")

Questions shape: (100, 2)
Question vectors shape: (100, 1024)


Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [ ]:
q_vec_df = questions_df[["question_id"]].copy()
q_vec_df["question_vector_json"] = [json.dumps([round(x, 4) for x in v.tolist()]) for v in question_vectors]
q_vec_df.to_csv("question-vectors.tsv", sep="\t", index=False)
print(f"Saved question-vectors.tsv with {len(q_vec_df)} rows")

Saved question-vectors.tsv with 100 rows


Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [ ]:
from sklearn.neighbors import NearestNeighbors

# Round vectors to match what is stored in paragraph-vectors.tsv.gz
paragraph_vectors_rounded = paragraph_vectors.round(4)
question_vectors_rounded = question_vectors.round(4)

nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn.fit(paragraph_vectors_rounded)

distances, indices = nn.kneighbors(question_vectors_rounded)
print(f"Found top 5 paragraph matches for {len(question_vectors_rounded)} questions")

Found top 5 paragraph matches for 100 questions


Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [ ]:
match_rows = []
for q_idx, question_id in enumerate(questions_df["question_id"]):
    for rank, para_idx in enumerate(indices[q_idx], start=1):
        match_rows.append({
            "question_id": question_id,
            "question_rank": rank,
            "document_title": parsed_df.iloc[para_idx]["document_title"],
            "paragraph_index": int(parsed_df.iloc[para_idx]["paragraph_index"])
        })

matches_df = pd.DataFrame(match_rows)
matches_df.to_csv("question-matches.tsv", sep="\t", index=False)
print(f"Saved question-matches.tsv with {len(matches_df)} rows")
matches_df.head(10)

Saved question-matches.tsv with 500 rows


,question_id,question_rank,document_title,paragraph_index
0,1,1,Tibet,10
1,1,2,Old_English,1
2,1,3,Bird_migration,5
3,1,4,"Punjab,_Pakistan",21
4,1,5,Symbiosis,8
5,4,1,BeiDou_Navigation_Satellite_System,18
6,4,2,BeiDou_Navigation_Satellite_System,7
7,4,3,BeiDou_Navigation_Satellite_System,12
8,4,4,BeiDou_Navigation_Satellite_System,3
9,4,5,BeiDou_Navigation_Satellite_System,13


Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [ ]:
# Get the first 5 questions sorted by question_id ascending
first5_qids = sorted(questions_df["question_id"].tolist())[:5]

# For each question, display all 5 matched paragraphs so we can identify the worst
print("Reviewing top-5 paragraph matches for first 5 questions:\n")
for qid in first5_qids:
    q_text = questions_df.loc[questions_df["question_id"] == qid, "question"].values[0]
    print(f"Question {qid}: {q_text}")
    top5 = matches_df[matches_df["question_id"] == qid].sort_values("question_rank")
    for _, row in top5.iterrows():
        para_row = parsed_df[
            (parsed_df["document_title"] == row["document_title"]) &
            (parsed_df["paragraph_index"] == row["paragraph_index"])
        ]
        context_snippet = para_row["paragraph_context"].values[0][:150]
        print(f"  Rank {row['question_rank']} | {row['document_title']} [para {row['paragraph_index']}]: {context_snippet}...")
    print()

# The worst match for each question is rank 5 (farthest Euclidean distance)
worst_rows = []
for qid in first5_qids:
    worst = matches_df[(matches_df["question_id"] == qid) & (matches_df["question_rank"] == 5)].iloc[0]
    worst_rows.append({
        "question_id": worst["question_id"],
        "document_title": worst["document_title"],
        "paragraph_index": worst["paragraph_index"]
    })

worst_df = pd.DataFrame(worst_rows)
worst_df.to_csv("worst-paragraphs.tsv", sep="\t", index=False)
print("Saved worst-paragraphs.tsv")
worst_df

Reviewing top-5 paragraph matches for first 5 questions:

Question 1: What was the goal of the abuse of region project?
  Rank 1 | Tibet [para 10]: The earliest Tibetan historical texts identify the Zhang Zhung culture as a people who migrated from the Amdo region into what is now the region of Gu...
  Rank 2 | Old_English [para 1]: The four main dialectal forms of Old English were Mercian, Northumbrian, Kentish, and West Saxon. Mercian and Northumbrian are together referred to as...
  Rank 3 | Bird_migration [para 5]: Aristotle noted that cranes traveled from the steppes of Scythia to marshes at the headwaters of the Nile. Pliny the Elder, in his Historia Naturalis,...
  Rank 4 | Punjab,_Pakistan [para 21]: The fairs held at the shrines of Sufi saints are called urs. They generally mark the death anniversary of the saint. On these occasions devotees assem...
  Rank 5 | Symbiosis [para 8]: An example of mutual symbiosis is the relationship between the ocellaris clownfish that dwell amo

  Rank 2 | Roman_Republic [para 47]: Clodius formed armed gangs that terrorised the city and eventually began to attack Pompey's followers, who in response funded counter-gangs formed by ...
  Rank 3 | Roman_Republic [para 45]: In 62 BC, Pompey returned victorious from Asia. The Senate, elated by its successes against Catiline, refused to ratify the arrangements that Pompey h...
  Rank 4 | Slavs [para 34]: ^10 Sub-groups of Slovenes include Prekmurians, Hungarian Slovenes, Carinthian Slovenes, Venetian Slovenes, Resians, and the extinct Carantanians and ...
  Rank 5 | Roman_Republic [para 26]: By 59 BC an unofficial political alliance known as the First Triumvirate was formed between Gaius Julius Caesar, Marcus Licinius Crassus, and Gnaeus P...

Question 13: What area is considered to have a desert climate despite having an annual monsoon season?
  Rank 1 | Mali [para 10]: Mali lies in the torrid zone and is among the hottest countries in the world. The thermal equator, which matches t

,question_id,document_title,paragraph_index
0,1,Symbiosis,8
1,4,BeiDou_Navigation_Satellite_System,13
2,7,Beyoncé,16
3,10,Roman_Republic,26
4,13,American_Idol,9


## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.